# Man-in-the-Middle — Passthrough Proxy

An attacker has man-in-the-middled the link between the GCS and the vehicle's
onboard Logic. The MITM sits as a transparent proxy:

```
telemetry:  Logic --MITM_TELEM--> [MITM] --GCS--> GCS
commands:   GCS   --MITM_CMD-->   [MITM] --GCS_CMD--> Logic
```

With the default `passthrough` strategy every message is forwarded unmodified,
byte-for-byte, in both directions — so the scenario behaves *identically* to
the no-MITM case. This is the baseline that the other attack strategies
(`blackout`, `hijack`, `spoof_gcs`, `spoof_owner_gcs`) build on.

There's no GCS intervention here — a plain AUTO mission is enough to prove
transparency, and it keeps this notebook focused on the proxy itself rather
than on the separate GCS-intervention feature (see the `5-GCS_intervention_*`
notebooks for that). Instead, this run opens the MITM's own terminal
(`terminals=[..., SimProcess.MITM]`) at `verbose=2`, so every message it
relays is printed live as `MITM downlink: forwarding <TYPE>` — for
`GLOBAL_POSITION_INT` specifically, the line also includes the actual
lat/lon/alt, so you can watch it match what the vehicle logic terminal
reports as the mission flies: same numbers, unmodified, just relayed.

Toggle the attack off by leaving `vehicle.mitm` as `None` (its default).

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import DATA_PATH, Color, Model
from simulator.entities import SimGCS, SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.helpers.processes import SimProcess
from simulator.planner import AutoPlan
from simulator.runtime.mitm import PassthroughStrategy
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin and waypoints

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

home = ENUPose(0, -20, 0, 0)
cruise_alt = 10.0  # m
model = Model.IRIS
sysid = 1

mission_wps = ENU.list([(0, 0, 0), (0, 0, cruise_alt), (0, 40, cruise_alt)])

## Vehicle + MITM

In [ ]:
mission_path = DATA_PATH / "missions" / "gcs_intervention.waypoints"

plan = AutoPlan.from_relative_path(
    name="north_mission",
    sysid=sysid,
    gra_origin=gra_origin,
    relative_home=home,
    relative_path=mission_wps,
    mission_path=str(mission_path),
    firmware=model.firmware,
)

vehicle = SimVehicle.from_relative(
    sysid=sysid,
    gcss=[SimGCS(name=f"{Color.BLUE.name}_{Color.BLUE.emoji}")],
    plan=plan,
    color=Color.BLUE,
    enu_origin=enu_origin,
    relative_home=home,
    relative_path=mission_wps,
    model=model,
    mitm=PassthroughStrategy(),
)


## Visualizer

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
gaz.markers.append(origin_marker)

## Oracle

In [ ]:
orac = Oracle()
orac.add_vehicle(vehicle)

# Simulator

In [ ]:
simulator = Simulator(
    oracle=orac,
    visualizer=gaz,
    verbose=2,
    terminals=[SimProcess.LOGIC, SimProcess.MITM],
)

simulator.preview()

In [ ]:
simulator.run()
